# Feature Engineering Pipeline for Payload-Byte Data

This notebook demonstrates the complete feature engineering pipeline including:
- Feature extraction (statistical, frequency, n-gram)
- Feature storage with HDF5
- Pipeline optimization for CPU environments
- Integration with ML pipelines

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import h5py

from src.data.feature_store import (
    FeatureStore, 
    create_statistical_features,
    create_frequency_features,
    create_ngram_features
)
from src.data.sample_data_generator import SampleDataGenerator
from src.data.pipeline_optimizer import PipelineOptimizer, create_optimized_pipeline

# Set random seed
np.random.seed(42)

# Create directories
Path('../data/features').mkdir(parents=True, exist_ok=True)

## 1. Generate Sample Data

First, let's generate sample packet data to work with.

In [ ]:
# Generate sample data
generator = SampleDataGenerator()

# Generate packets with different characteristics
n_samples = 1000
packets, labels = generator.generate_packet_data(
    n_samples=n_samples,
    benign_ratio=0.7,
    packet_size_range=(256, 1500)
)

# Convert to fixed-size array for feature extraction
max_length = 1500
packet_array = np.zeros((n_samples, max_length), dtype=np.uint8)

for i, packet in enumerate(packets):
    length = min(len(packet), max_length)
    packet_array[i, :length] = packet[:length]

print(f"Generated {n_samples} packets")
print(f"Packet array shape: {packet_array.shape}")
print(f"Label distribution: Benign={sum(labels == 0)}, Malicious={sum(labels == 1)}")

## 2. Statistical Feature Extraction

Extract basic statistical features from packet data.

In [ ]:
# Extract statistical features
print("Extracting statistical features...")
start_time = time.time()

stat_features = create_statistical_features(packet_array)

elapsed_time = time.time() - start_time
print(f"Extraction completed in {elapsed_time:.2f} seconds")
print(f"Statistical features shape: {stat_features.shape}")
print(f"\nFeature names: {list(stat_features.columns)}")

# Display sample features
stat_features.head()

## 3. Frequency Domain Features

Extract frequency domain features using FFT.

In [ ]:
# Extract frequency features
print("Extracting frequency domain features...")
start_time = time.time()

freq_features = create_frequency_features(packet_array, n_components=20)

elapsed_time = time.time() - start_time
print(f"Extraction completed in {elapsed_time:.2f} seconds")
print(f"Frequency features shape: {freq_features.shape}")

# Visualize frequency components
plt.figure(figsize=(12, 4))

# Plot average frequency components for each class
benign_freq = freq_features[labels == 0].mean()
malicious_freq = freq_features[labels == 1].mean()

freq_cols = [col for col in freq_features.columns if 'freq_component' in col]
x = range(len(freq_cols))

plt.plot(x, benign_freq[freq_cols], 'b-', label='Benign', alpha=0.7)
plt.plot(x, malicious_freq[freq_cols], 'r-', label='Malicious', alpha=0.7)
plt.xlabel('Frequency Component')
plt.ylabel('Average Magnitude')
plt.title('Average Frequency Components by Class')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. N-gram Features

Extract n-gram features from byte sequences.

In [ ]:
# Extract n-gram features
print("Extracting n-gram features...")
start_time = time.time()

ngram_features = create_ngram_features(packet_array[:500], n=2, top_k=30)  # Subset for speed

elapsed_time = time.time() - start_time
print(f"Extraction completed in {elapsed_time:.2f} seconds")
print(f"N-gram features shape: {ngram_features.shape}")

# Visualize top n-grams
ngram_sums = ngram_features.sum()
top_ngrams = ngram_sums.nlargest(10)

plt.figure(figsize=(10, 6))
top_ngrams.plot(kind='barh')
plt.xlabel('Total Count')
plt.title('Top 10 Most Common Byte N-grams')
plt.tight_layout()
plt.show()

## 5. Feature Store Integration

Store extracted features in the HDF5-based feature store.

In [ ]:
# Initialize feature store
store_path = Path('../data/features/feature_store.h5')
feature_store = FeatureStore(store_path)

# Store statistical features
stat_key = feature_store.store_features(
    stat_features,
    name='statistical_features',
    description='Basic statistical features from packet bytes',
    preprocessing_params={'max_length': max_length}
)
print(f"Stored statistical features: {stat_key}")

# Store frequency features
freq_key = feature_store.store_features(
    freq_features,
    name='frequency_features',
    description='FFT-based frequency domain features',
    preprocessing_params={'n_components': 20}
)
print(f"Stored frequency features: {freq_key}")

# Get feature statistics
stats = feature_store.get_feature_statistics()
print(f"\nTotal feature store size: {stats['total_size_mb']:.2f} MB")
print(f"Total features: {stats['total_features']}")
print(f"Total samples: {stats['total_samples']}")

## 6. Feature Pipeline Creation

Create an end-to-end feature extraction pipeline.

In [ ]:
# Create feature pipeline
pipeline = feature_store.create_feature_pipeline({
    'name': 'comprehensive_features',
    'version': '1.0'
})

# Add extractors
pipeline.add_extractor(create_statistical_features, 'stats')
pipeline.add_extractor(create_frequency_features, 'freq', {'n_components': 15})

# Process new data through pipeline
new_packets = packet_array[800:900]  # Use subset for demo

print("Processing data through feature pipeline...")
start_time = time.time()

feature_key = pipeline.extract_and_store(
    new_packets,
    feature_set_name='pipeline_features',
    version='demo_v1'
)

elapsed_time = time.time() - start_time
print(f"Pipeline processing completed in {elapsed_time:.2f} seconds")
print(f"Stored features: {feature_key}")

# Load and verify features
loaded_features, metadata = feature_store.load_features('pipeline_features', 'demo_v1')
print(f"\nLoaded features shape: {loaded_features.shape}")
print(f"Created at: {metadata.created_at}")

## 7. Pipeline Optimization Analysis

Analyze and optimize pipeline performance for CPU-only environments.

In [ ]:
# Initialize optimizer (CPU-only mode)
optimizer = PipelineOptimizer(use_gpu=False)

# Define a sample processing function
def process_packets(batch):
    """Sample processing function for optimization testing"""
    # Convert list to array if needed
    if isinstance(batch, list):
        batch_array = np.zeros((len(batch), max_length), dtype=np.uint8)
        for i, packet in enumerate(batch):
            length = min(len(packet), max_length)
            batch_array[i, :length] = packet[:length]
    else:
        batch_array = batch
        
    # Extract features
    stats = create_statistical_features(batch_array)
    return stats.values

# Test different batch sizes
print("Optimizing batch size...")
batch_optimization = optimizer.optimize_batch_size(
    process_packets,
    packets[:500],  # Use subset for testing
    batch_sizes=[8, 16, 32, 64],
    target_memory_mb=500
)

print(f"\nOptimal batch size: {batch_optimization['optimal_batch_size']}")
print(f"Recommendation: {batch_optimization['recommendation']}")

# Visualize results
results_df = pd.DataFrame([
    {
        'batch_size': r['batch_size'],
        'throughput': r['metrics'].throughput_samples_per_sec,
        'memory_mb': r['metrics'].memory_usage_mb,
        'cpu_percent': r['metrics'].cpu_usage_percent
    }
    for r in batch_optimization['all_results']
])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Throughput
axes[0].plot(results_df['batch_size'], results_df['throughput'], 'o-')
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Throughput (samples/sec)')
axes[0].set_title('Processing Throughput')
axes[0].grid(True, alpha=0.3)

# Memory usage
axes[1].plot(results_df['batch_size'], results_df['memory_mb'], 'o-', color='orange')
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Memory Usage (MB)')
axes[1].set_title('Memory Consumption')
axes[1].grid(True, alpha=0.3)

# CPU usage
axes[2].plot(results_df['batch_size'], results_df['cpu_percent'], 'o-', color='green')
axes[2].set_xlabel('Batch Size')
axes[2].set_ylabel('CPU Usage (%)')
axes[2].set_title('CPU Utilization')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Parallel Processing Benchmark

Test parallel processing performance with different worker configurations.

In [ ]:
# Benchmark parallel processing
print("Benchmarking parallel processing...")

parallel_results = optimizer.benchmark_parallel_processing(
    process_packets,
    packets[:200],  # Smaller subset for demo
    worker_counts=[1, 2, 4],
    batch_size=32
)

print(f"\nOptimal configuration: {parallel_results['recommendation']}")

# Visualize speedup
workers = [r['workers'] for r in parallel_results['all_results']]
speedups = [r['speedup'] for r in parallel_results['all_results']]
throughputs = [r['metrics'].throughput_samples_per_sec for r in parallel_results['all_results']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Speedup plot
ax1.plot(workers, speedups, 'o-', linewidth=2, markersize=8)
ax1.plot(workers, workers, '--', alpha=0.5, label='Ideal speedup')
ax1.set_xlabel('Number of Workers')
ax1.set_ylabel('Speedup Factor')
ax1.set_title('Parallel Processing Speedup')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Throughput plot
ax2.bar(workers, throughputs, alpha=0.7, color='green')
ax2.set_xlabel('Number of Workers')
ax2.set_ylabel('Throughput (samples/sec)')
ax2.set_title('Processing Throughput by Worker Count')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Feature Quality Analysis

Analyze the quality and discriminative power of extracted features.

In [ ]:
# Combine all features
all_features = pd.concat([stat_features, freq_features], axis=1)

# Calculate feature importance using mutual information
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(all_features, labels[:len(all_features)])
mi_df = pd.DataFrame({
    'feature': all_features.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

# Plot top features
plt.figure(figsize=(10, 8))
top_features = mi_df.head(20)
plt.barh(range(len(top_features)), top_features['mi_score'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Mutual Information Score')
plt.title('Top 20 Most Informative Features')
plt.tight_layout()
plt.show()

# Feature correlation analysis
plt.figure(figsize=(12, 10))
correlation_matrix = all_features.corr()
mask = np.triu(np.ones_like(correlation_matrix), k=1)
sns.heatmap(correlation_matrix, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": .8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 10. Production Configuration

Generate optimized configuration for production deployment.

In [ ]:
# Create optimized pipeline configuration
data_path = Path('../data/processed/payload_byte_data.h5')

# Create dummy file for configuration generation
data_path.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(data_path, 'w') as f:
    f.create_dataset('dummy', data=np.zeros((100, 100)))

production_config = create_optimized_pipeline(
    data_path=data_path,
    target_memory_mb=4000,
    use_gpu=False  # CPU-only for development
)

print("Production Pipeline Configuration:")
print("=" * 50)
for key, value in production_config.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

# Save configuration
import json
config_path = Path('../configs/optimized_pipeline_config.json')
config_path.parent.mkdir(exist_ok=True)

with open(config_path, 'w') as f:
    json.dump(production_config, f, indent=2)
print(f"\nConfiguration saved to: {config_path}")

# Clean up dummy file
data_path.unlink()

## Summary

### Key Achievements:

1. **Feature Extraction Pipeline**:
   - Statistical features (mean, std, entropy, etc.)
   - Frequency domain features (FFT components)
   - N-gram features for byte patterns

2. **Feature Storage**:
   - HDF5-based feature store with versioning
   - Efficient storage and retrieval
   - Metadata tracking

3. **Performance Optimization**:
   - Batch size optimization
   - Parallel processing benchmarks
   - Memory usage analysis

4. **Production Ready**:
   - Optimized configuration for CPU environments
   - Scalable architecture
   - Performance monitoring

### Next Steps:

1. Integrate with ML models for feature validation
2. Add more sophisticated feature extractors
3. Implement real-time feature extraction
4. Deploy to cloud with GPU support